# Benchmark Viewer

Displays the **question**, **available functions**, and **ground-truth** answer
for every entry in a benchmark file. Point `BENCHMARK_FILE` at any benchmark
under `data/benchmarks/<CATEGORY>/` (e.g. `he_translatable_full_shuffled.json`,
`he_translatable_query_shuffled.json`, `he_translatable_full.json`, `eng_base.json`)
and run all cells.

Files may live either directly under the category or inside a locale subfolder
(e.g. `heb/`); the subfolder is auto-detected, or set `SUBDIR` explicitly. The
matching ground-truth answers are read from the file of the same name under
`possible_answer/` (mirroring the same subfolder). Hebrew / Arabic text is
rendered right-to-left automatically.


In [14]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import json
from pathlib import Path

from IPython.display import display, HTML

NOTEBOOK_DIR = Path().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT    = PACKAGE_ROOT / "data" / "benchmarks"

print(f"Package root: {PACKAGE_ROOT}")


Package root: C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl


In [15]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
CATEGORY = "multiple"   # benchmark folder under data/benchmarks/

# Which benchmark file to view (a basename). Its ground truth is read from the
# file of the same name under possible_answer/.
BENCHMARK_FILE = "he_translatable_query_shuffled.json"

# Locale subfolder under the category (e.g. "heb"). Leave as None to auto-detect;
# set "" to force the flat layout (files directly under the category).
SUBDIR = None

# Show only entries whose id contains this string (e.g. "multiple_17"), or None:
FILTER_id = None

# Maximum number of entries to display (None = all):
MAX_DISPLAY = None


In [16]:
# ── Cell 3: Load benchmark + ground truth ─────────────────────────────────────
bench_root = DATA_ROOT / CATEGORY

# Resolve the (optional) locale subfolder: files may sit directly under the
# category or inside a subfolder like 'heb/'.
if SUBDIR is None:
    if (bench_root / BENCHMARK_FILE).exists():
        subdir = ""
    else:
        cands = [d.name for d in sorted(bench_root.iterdir())
                 if d.is_dir() and d.name != "possible_answer"
                 and (d / BENCHMARK_FILE).exists()]
        subdir = cands[0] if cands else ""
        if subdir:
            print(f"Auto-detected locale subfolder: {subdir!r}")
else:
    subdir = SUBDIR

bench_dir   = bench_root / subdir if subdir else bench_root
answer_dir  = (bench_root / "possible_answer" / subdir) if subdir else (bench_root / "possible_answer")
source_path = bench_dir / BENCHMARK_FILE
answer_path = answer_dir / BENCHMARK_FILE

if not source_path.exists():
    raise FileNotFoundError(f"Missing benchmark file: {source_path}")

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

entries = load_jsonl(source_path)   # list, original order preserved

answers_by_id = {}
if answer_path.exists():
    for a in load_jsonl(answer_path):
        answers_by_id[a["id"]] = a
else:
    print(f"[note] no ground-truth file at {answer_path} — answers will show as 'no answer'.")

def find_answer(entry):
    """Match an entry to its answer, tolerating the bare vs. locale-suffixed
    id conventions (possible_answer files sometimes use 'multiple_2' and
    sometimes 'multiple_2_he')."""
    sid = entry.get("source_id")
    loc = entry.get("locale")
    candidates = [entry.get("id"), sid]
    if sid and loc:
        candidates.append(f"{sid}_{loc}")
    for c in candidates:
        if c in answers_by_id:
            return answers_by_id[c]
    return None

levels = sorted({e.get("localization_level", "?") for e in entries})
locales = sorted({e.get("locale", "?") for e in entries})
print(f"File              : {source_path.relative_to(DATA_ROOT)}")
print(f"Entries           : {len(entries)}")
print(f"Ground-truth file : {answer_path.relative_to(DATA_ROOT) if answer_path.exists() else '(none)'}")
print(f"Locale(s)         : {locales}")
print(f"Localization level: {levels}")


Auto-detected locale subfolder: 'heb'
File              : multiple\heb\he_translatable_query_shuffled.json
Entries           : 132
Ground-truth file : multiple\possible_answer\heb\he_translatable_query_shuffled.json
Locale(s)         : ['he']
Localization level: ['query_shuffled']


In [17]:
# ── Cell 4: Render ────────────────────────────────────────────────────────────
CARD_BG     = "#1e1e1e"
CARD_BORDER = "#3a3a3a"
SECTION_BG  = "#2a2a2a"
TITLE_COLOR = "#e0e0e0"
LABEL_COLOR = "#aaaaaa"
TEXT_COLOR  = "#d4d4d4"
USER_COLOR  = "#7eb8f7"
ASST_COLOR  = "#aaaaaa"

def format_query(entry):
    parts = []
    for turn in entry.get("question", []):
        for msg in turn:
            role  = msg.get("role", "?")
            color = USER_COLOR if role == "user" else ASST_COLOR
            parts.append(
                f'<div style="margin-bottom:4px"><span style="color:{color};font-weight:600">{role}:</span> '
                f'<span dir="auto" style="color:{TEXT_COLOR}">{msg.get("content","")}</span></div>'
            )
    return "".join(parts) or "<i style='color:#888'>no question</i>"

def format_functions(entry):
    lines = []
    for func in entry.get("function", []):
        name   = func.get("name", "?")
        params = func.get("parameters", {}).get("properties", {})
        req    = set(func.get("parameters", {}).get("required", []))
        param_strs = []
        for p, pdef in params.items():
            star = "*" if p in req else ""
            extra = ""
            if pdef.get("enum"):
                extra = ' <span style="color:#888">[' + " | ".join(map(str, pdef["enum"])) + "]</span>"
            param_strs.append(
                f'<code style="color:#ce9178">{p}{star}</code>'
                f'<span style="color:#888">: {pdef.get("type","any")}</span>{extra}'
            )
        lines.append(f'<div style="margin-bottom:4px"><b style="color:#dcdcaa">{name}</b>({", ".join(param_strs)})</div>')
    return "".join(lines) or "<i style='color:#888'>no functions</i>"

def format_ground_truth(answer):
    if not answer:
        return "<i style='color:#888'>no answer</i>"
    lines = []
    for call in answer.get("ground_truth", []):
        for func_name, params in call.items():
            lines.append(f'<div style="margin-top:4px"><b style="color:{TITLE_COLOR}">{func_name}</b></div>')
            for param, values in params.items():
                vals_html = " <span style='color:#666'>|</span> ".join(
                    f'<code dir="auto" style="color:#b5cea8">{json.dumps(v, ensure_ascii=False)}</code>'
                    for v in values
                )
                lines.append(
                    f'<div>&nbsp;&nbsp;<code style="color:#ce9178">{param}</code> = {vals_html}</div>'
                )
    return "".join(lines)

def render_card(entry):
    answer = find_answer(entry)
    label  = entry.get("id", "?")
    level  = entry.get("localization_level", "")
    tag = (f'<span style="font-size:0.75em;color:#888;font-weight:400;'
           f'border:1px solid {CARD_BORDER};border-radius:4px;padding:1px 6px;'
           f'margin-left:8px">{level}</span>') if level else ""
    return f"""
<div style="border:1px solid {CARD_BORDER};border-radius:8px;
            padding:16px;margin-bottom:14px;font-family:sans-serif;background:{CARD_BG}">
  <div style="font-size:1.05em;font-weight:700;color:{TITLE_COLOR};margin-bottom:10px">{label}{tag}</div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;font-size:0.9em">
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">QUERY</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.6">{format_query(entry)}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">FUNCTIONS</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{format_functions(entry)}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">GROUND TRUTH</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{format_ground_truth(answer)}</div>
    </div>
  </div>
</div>"""

shown = entries
if FILTER_id is not None:
    shown = [e for e in shown if FILTER_id in e.get("id", "")]
if MAX_DISPLAY:
    shown = shown[:MAX_DISPLAY]

print(f"Showing {len(shown)} of {len(entries)} entries.")
display(HTML("".join(render_card(e) for e in shown)))


Showing 132 of 132 entries.
